In [1]:
import numpy as np
import polars as pl

In [2]:
golden_set = pl.read_parquet("data/golden_set.parquet").filter(pl.col("geonameIds").list.len() > 0)

In [3]:
import httpx
from tqdm.auto import tqdm


def search_batch(queries: list[str], top_k: int = 50):
    base_url = "http://localhost:8000/v1/search"
    results = []
    
    with httpx.Client(timeout=30.0) as client:  # синхронный клиент
        for query in tqdm(queries):
            response = client.get(base_url, params={"text": query, "top_k": top_k})
            response.raise_for_status()
            results.append(response.json())
    
    return results

In [4]:
from ir_measures import P, Recall, RR, calc


qrels_dict = {}
for row in golden_set.iter_rows(named=True):
    qrels_dict.update({row["query"]: {str(gid): 1 for gid in row["geonameIds"]}})

predictions = search_batch(qrels_dict.keys(), top_k=50)

  0%|          | 0/180 [00:00<?, ?it/s]

In [5]:
run_dict = {}
for pred in predictions:
    run_dict.update({pred["query"]: {str(r["geonameid"]): float(np.log10(r["population"] + 1)) for r in  pred["results"]}})

metrics = calc([RR, P@1, Recall@5, Recall@25, Recall@50], qrels_dict, run_dict)
metrics_aggregated = pl.DataFrame({str(k): v for k, v in metrics.aggregated.items()}).unpivot().sort("value")

metrics_per_query = {str(m): {"query": [], "value": []} for m in metrics.aggregated}

for metric in metrics.per_query:
    mname = str(metric.measure)
    metrics_per_query[mname]["query"].append(metric.query_id)
    metrics_per_query[mname]["value"].append(metric.value)

for mname in metrics_per_query:
    metrics_per_query[mname] = pl.DataFrame(metrics_per_query[mname]).sort("value")

In [6]:
metrics_aggregated

variable,value
str,f64
"""P@1""",0.85
"""R@5""",0.883513
"""RR""",0.902044
"""R@25""",0.951612
"""R@50""",0.965498


In [7]:
for k in metrics_per_query:
    print(k)
    display(metrics_per_query[k].limit(5))

RR


query,value
str,f64
"""Подскажите, какую музыку можно…",0.0
"""Ağrı Dağı'na tırmanmak için en…",0.0
"""St. Louis'de Gateway Kemeri'ni…",0.076923
"""Что посмотреть в Санта-Фе за д…",0.090909
"""Какие цены на недвижимость в С…",0.166667


R@50


query,value
str,f64
"""Подскажите, какую музыку можно…",0.0
"""Ağrı Dağı'na tırmanmak için en…",0.0
"""Van, Sinop ve Şanlıurfa’dan bi…",0.5
"""Какая погода в Ване в конце ок…",0.5
"""Где в Кливленде можно посмотре…",0.538462


R@5


query,value
str,f64
"""Какие цены на недвижимость в С…",0.0
"""Подскажите, какую музыку можно…",0.0
"""St. Louis'de Gateway Kemeri'ni…",0.0
"""Ağrı Dağı'na tırmanmak için en…",0.0
"""Что посмотреть в Санта-Фе за д…",0.0


P@1


query,value
str,f64
"""Какой самый дешевый способ доб…",0.0
"""Что посмотреть в Сент-Луисе за…",0.0
"""Какие цены на недвижимость в С…",0.0
"""Anchorage'da kış aylarında Aur…",0.0
"""Какая погода ожидается в Чите …",0.0


R@25


query,value
str,f64
"""Подскажите, какую музыку можно…",0.0
"""Ağrı Dağı'na tırmanmak için en…",0.0
"""Куда лучше поехать на выходные…",0.25
"""Сколько лететь из Портленда в …",0.25
"""Где в Кливленде можно посмотре…",0.307692


In [8]:
no_city_df = pl.read_parquet("data/golden_set.parquet").filter(pl.col("geonameIds").list.len() == 0)

In [9]:
results = search_batch(no_city_df["query"].to_list())

  0%|          | 0/55 [00:00<?, ?it/s]

In [11]:
results = [r["results"] for r in results]

In [12]:
accuracy = 0
for r in results:
    if r == []:
        accuracy += 1
accuracy /= len(results)

In [13]:
accuracy

0.9272727272727272